In [10]:
import dynamiqs as dq
import jax.numpy as jnp
import jax

In [3]:
n = 16
omega = 1.0
kappa = 0.1
alpha0 = 1.0
T = 2 * jnp.pi

a = dq.destroy(n)
H = omega*a.dag()@a
jump_ops = [jnp.sqrt(kappa)*a]
psi0 = dq.coherent(n, alpha0)
tsave = jnp.linspace(0, T, 101)

result = dq.mesolve(H, jump_ops, psi0, tsave)

|██████████| 100.0% ◆ elapsed 13.45ms ◆ remaining 0.00ms


In [4]:
print(result)

==== MESolveResult ====
Method : Tsit5
Infos  : 40 steps (40 accepted, 0 rejected)
States : QArray complex64 (101, 16, 16) | 202.0 Kb


In [9]:
def population(omega, kappa, alpha0):
    a  = dq.destroy(n)
    H = omega*a.dag()@a
    jump_ops = [jnp.sqrt(kappa)*a]
    psi0 = dq.coherent(n, alpha0)
    tsave = jnp.linspace(0, T, 101)

    result = dq.mesolve(H, jump_ops, psi0, tsave)

    return dq.expect(dq.number(n), result.states[-1]).real



grad_population = jax.grad(population, argnums=(0,1,2))

grads = grad_population(omega, kappa, alpha0)

print(grads)





|██████████| 100.0% ◆ elapsed 9.46ms ◆ remaining 0.00ms


(Array(0., dtype=float32, weak_type=True), Array(-3.3520043, dtype=float32, weak_type=True), Array(1.0669758, dtype=float32, weak_type=True))


In [10]:
print(grads[0])
print(grads[1])
print(grads[2])






0.0
-3.3520043
1.0669758


In [ ]:
def make_flat_top_pulse(
    net_duration_ns,
    flat_top_duration_ns,
    rise_time_ns,
    sigma_ns,
    sigma_ns_fall = None,
    fall_time_ns = None,
    dt_ns = 1
    ):
    
    if fall_time_ns is None:
        fall_time_ns = rise_time_ns
    if sigma_ns_fall is None:
        sigma_ns_fall = sigma_ns
    flat_top_duration_ns = flat_top_duration_ns[:,None]

    #gaussian rise half
    flat_start_ns = rise_time_ns
    flat_end_ns = flat_start_ns + flat_top_duration_ns
    pulse_end_ns = flat_end_ns + fall_time_ns
    print(jnp.shape(flat_top_duration_ns))
    # print(jnp.shape(flat_top_duration_ns))
    total_len = int(round(net_duration_ns/dt_ns))
    t = jnp.arange(total_len)*dt_ns
    gaussian_rise_pulse = lambda t: jnp.exp(-(t-rise_time_ns)**2/(2*sigma_ns**2))
    gaussian_fall_pulse = lambda t: jnp.exp(-(t-flat_end_ns)**2/(2*sigma_ns_fall**2))

    pulse = jnp.zeros(total_len)
    pulse = jnp.where(t < rise_time_ns, gaussian_rise_pulse(t),  #jnp.where(condition, true_action, false_action)
              jnp.where(t <= flat_end_ns, 1,
                        jnp.where(t <= pulse_end_ns, gaussian_fall_pulse(t), 0)))


    
    return pulse







    
        
        


In [32]:
total_len_array = jnp.linspace(0, 120, 120)
flat_top_duration = jnp.array([50, 55, 60])
flat_top_duration = flat_top_duration[:,None]
print(jnp.shape(total_len_array))
print(jnp.shape(flat_top_duration))
sweep_array = flat_top_duration*total_len_array
print(jnp.shape(sweep_array))


(120,)
(3, 1)
(3, 120)


In [34]:
flattop = make_flat_top_pulse(net_duration_ns=120, flat_top_duration_ns=jnp.array([50, 55, 60]), rise_time_ns=20, sigma_ns=20/4)
jnp.shape(flattop)


(3, 1)


(3, 120)

In [8]:
a = [0,
    1]
a = jnp.array(a)

b = [
    0,
    0,
    1
]
b = jnp.array(b)

tp = a*b

TypeError: mul got incompatible shapes for broadcasting: (2,), (3,).

In [2]:
func_lambda(8)

64

In [12]:
#lets define the hamiltionian for the two qubit system
# we move in the frame of the control or the target?
# lets take the control qubits frame
from fourqubitv5.Configuration_Files.config_dictionaries import fq_vals
import numpy as np
import dynamiqs as dq
import jax.numpy as jnp
f_q_tgt = fq_vals['fq_vals']['2']
f_q_ctrl = fq_vals['fq_vals']['1']
delta_fq = np.abs(f_q_tgt - f_q_ctrl)
#in MHz
n_levels = 3

a_tgt = dq.destroy(n_levels)
a_ctrl = dq.destroy(n_levels)
amp_v = None
sigma_v = None
ph_v = None
t_rise_v = None
t_hold_v = None

pulse_params = {'amp_mhz': amp_v, 'gaussian_sigma': sigma_v, 'phase_rad': ph_v,
                't_rise_ns': t_rise_v, 't_hold': t_hold_v}







2026-07-01 11:58:55,601 - qm - INFO     - Starting session: 121bc861-7fe0-4f6a-a006-b87cb7068668


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

def envelope(t, rise, flat, fall, sigma, sigma_f):
    c1 = rise
    c2 = rise + flat
    tail = c2 + fall
    e = np.where(
        t < c1,
        np.exp(-0.5 * ((t - c1) / sigma) ** 2),
        np.where(t <= c2, 1.0, np.exp(-0.5 * ((t - c2) / sigma_f) ** 2)),
    )
    return np.where(t > tail, 0.0, np.where(t < 0, 0.0, e))

def plot(net=140, flat=60, rise=28, sigma=9.0, fall=28, sigma_f=9.0, dt=1):
    c1, c2, tail = rise, rise + flat, rise + flat + fall
    tt = np.linspace(0, net, 2000)              # smooth curve
    yy = envelope(tt, rise, flat, fall, sigma, sigma_f)

    L = max(1, round(net / dt))
    ts = np.arange(L) * dt                       # sampled grid
    ys = envelope(ts, rise, flat, fall, sigma, sigma_f)

    a0 = float(envelope(np.array([0.0]), rise, flat, fall, sigma, sigma_f))
    aend = float(envelope(np.array([tail - 1e-6]), rise, flat, fall, sigma, sigma_f))

    fig, ax = plt.subplots(figsize=(9, 3.5))
    # region shading
    ax.axvspan(0, min(c1, net),        color="#AFA9EC", alpha=0.18, lw=0)
    ax.axvspan(min(c1, net), min(c2, net), color="#5DCAA5", alpha=0.18, lw=0)
    ax.axvspan(min(c2, net), min(tail, net), color="#AFA9EC", alpha=0.18, lw=0)
    if tail < net:
        ax.axvspan(tail, net, color="#888780", alpha=0.10, lw=0)
    # sampled staircase (what the OPX plays)
    ax.step(ts, ys, where="post", color="#888780", alpha=0.55, lw=1, label=f"sampled dt={dt}ns")
    # continuous envelope
    ax.plot(tt, yy, color="#378ADD", lw=2.5, label="continuous")
    # boundaries + start-amp dot
    for x, lab in [(c1, "c1"), (c2, "c2")]:
        if x <= net:
            ax.axvline(x, color="0.5", ls="--", lw=0.8)
            ax.text(x, 1.11, lab, ha="center", fontsize=9, color="0.4")
    ax.plot(0, a0, "o", color="#D85A30", ms=6)

    ax.set(xlim=(0, net), ylim=(0, 1.15), xlabel="t (ns)", ylabel="amplitude")
    ax.set_title(
        f"start={a0:.3f}   end={aend:.3f}   extent={tail:.0f}ns   L={L}",
        fontsize=10,
    )
    if tail > net:
        ax.text(0.5, 0.5, "fall clipped by net_duration", transform=ax.transAxes,
                ha="center", color="#BA7517", fontsize=11)
    elif a0 > 0.02:
        ax.text(0.02, 0.9, f"start amp {a0:.3f}: discontinuous jump from 0",
                transform=ax.transAxes, color="#BA7517", fontsize=9)
    ax.legend(loc="upper right", fontsize=8)
    plt.tight_layout()
    plt.show()

S = lambda lo, hi, st, v, d: widgets.FloatSlider(
    min=lo, max=hi, step=st, value=v, description=d,
    continuous_update=True, style={"description_width": "140px"},
    layout=widgets.Layout(width="420px"))

widgets.interact(
    plot,
    net=S(40, 320, 1, 140, "net_duration_ns"),
    flat=S(0, 200, 1, 60, "flat_top_duration_ns"),
    rise=S(2, 80, 1, 28, "rise_time_ns"),
    sigma=S(1, 30, 0.5, 9, "sigma_ns"),
    fall=S(2, 80, 1, 28, "fall_time_ns"),
    sigma_f=S(1, 30, 0.5, 9, "sigma_ns_fall"),
    dt=S(1, 8, 1, 1, "dt_ns"),
);

interactive(children=(FloatSlider(value=140.0, description='net_duration_ns', layout=Layout(width='420px'), ma…